# Animate a 2D mesh, then a 3D mesh

Select your **Python 3.13.6** kernel and choose **Run All**. Two short, looping
animations will appear directly below their cells. No datasets, external render
window, or interactive Jupyter rendering backend are needed.

Open this notebook in this branch's repository root so `from exploration import
TimeController` works. Mesh creation and plotting use public PyVista APIs;
`TimeController` is this project's small playback helper.

If dependencies are missing, run this in a terminal using your kernel's Python,
then restart the kernel:

```sh
python -m pip install pyvista imageio pillow ipykernel
```

The code has no exact Python version requirement. Newer Python versions also need
compatible PyVista, NumPy, and VTK packages. As of October 22, 2026,
[Python 3.15 is the newest stable release](https://www.python.org/downloads/),
but the [published VTK wheels](https://pypi.org/project/vtk/#files) cover Python
through 3.14, not 3.15. Use 3.13.6 or a supported 3.14 environment until those
dependencies support 3.15. Rendering requires a working OpenGL backend, even
when no window is shown.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import tempfile
from typing import TYPE_CHECKING

from IPython.display import Image
import numpy as np

from exploration import TimeController
import pyvista as pv

if TYPE_CHECKING:
    from collections.abc import Callable

pv.OFF_SCREEN = True
sys.version.split()[0], pv.__version__

## One small display function

Each object supplies `make_frame(index)`. This function uses the same controller
as saved-result playback, renders each sample, and returns an inline GIF.
Temporary GIF files are deleted automatically, and the plotter closes even if
rendering fails.

There are 24 sample times in one cycle. `fps=10` controls playback speed separately
from these sample times. The slider is hidden in the exported animation.

In [ ]:
times = np.linspace(0.0, 1.0, 24, endpoint=False)


def animate(make_frame: Callable[[int], pv.DataSet], *, view: str) -> Image:
    """Render one cycle and display it inline without keeping result files."""
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / 'animation.gif'
        pl = pv.Plotter(notebook=False, window_size=(480, 360))
        try:
            controller = TimeController(pl, times, make_frame, scalars='value', clim=(-1, 1))
            pl.clear_slider_widgets()
            pl.set_background('white')
            pl.camera_position = view
            pl.enable_parallel_projection()
            pl.camera.zoom(0.7)
            pl.open_gif(path, fps=10)
            pl.show(interactive=False, auto_close=False)
            for index in range(len(times)):
                controller.select_index(index)
                pl.write_frame()
        finally:
            pl.close()
        return Image(data=path.read_bytes(), format='gif')

## 1. A 2D object

Create a flat plane with `pv.Plane()`. It stretches in the XY plane while a
colored wave travels across it. All Z coordinates remain zero; `view='xy'`
looks straight down at the plane.

In [ ]:
def plane_frame(index: int) -> pv.PolyData:
    """Create one flat, deforming mesh at the requested sample time."""
    phase = 2 * np.pi * times[index]
    mesh = pv.Plane(i_resolution=32, j_resolution=24)
    mesh.points[:, 0] *= 1 + 0.2 * np.sin(phase)
    mesh.points[:, 1] *= 1 - 0.2 * np.sin(phase)
    mesh['value'] = np.sin(2 * np.pi * mesh.points[:, 0] - phase)
    return mesh


animate(plane_frame, view='xy')

## 2. A 3D object

Create a sphere with `pv.Sphere()` and deform it along three axes. The oblique
`view='iso'` makes its changing 3D shape visible. Colors show the Z coordinate.

In [ ]:
def sphere_frame(index: int) -> pv.PolyData:
    """Create one three-dimensional mesh at the requested sample time."""
    phase = 2 * np.pi * times[index]
    mesh = pv.Sphere(theta_resolution=32, phi_resolution=24)
    mesh.points *= np.array(
        [1 + 0.2 * np.sin(phase), 1 - 0.15 * np.sin(phase), 1 + 0.1 * np.cos(phase)]
    )
    mesh['value'] = mesh.points[:, 2]
    return mesh


animate(sphere_frame, view='iso')

These are synthetic geometry demonstrations, not heart or crash simulations.
Frames are created on demand; no sequence of meshes is preloaded. The GIF encoder
can buffer rendered images, so the examples use small meshes and 24 small frames.
GIFs are visual previews, not storage for scientific results.

For existing simulation files, use `ReaderSource.frame` in place of these mesh
functions. See [saved-result playback](exploration/READER_PLAYBACK.md) for explicit
spatial-part and scalar selection. No temporal interpolation is used.